# BookWise — E-Commerce Book Pricing & Rating Analytics

## 01. Web Data Acquisition

### Source
**Books to Scrape** — https://books.toscrape.com/

Books to Scrape is a publicly accessible website designed for web-scraping practice. The catalog can be accessed without authentication or a user account.

### Acquisition Method
The dataset is collected programmatically using Python with:

- `requests` for HTTP requests
- `BeautifulSoup` for HTML parsing
- `pandas` for structured data storage

The acquisition process consists of two stages:

1. Scrape the catalogue/listing pages to obtain book-level information and product URLs.
2. Visit each individual product page to enrich the records with additional attributes.

The final acquisition target is at least 1,000 unique book records.

### Project Objective

The objective of this project is to analyze an online bookstore's product catalog to understand book pricing, ratings, availability, and category-level patterns, and to develop data-driven insights for better retail decision-making.

This notebook focuses on collecting publicly available book data from the web in a reproducible manner.

In [6]:
BASE_URL = "https://books.toscrape.com/"
CATALOGUE_URL = BASE_URL + "catalogue/"

REQUEST_TIMEOUT = 20
REQUEST_DELAY = 0.5
MIN_REQUIRED_RECORDS = 1000

print("Source:", BASE_URL)
print("Minimum required records:", MIN_REQUIRED_RECORDS)

Source: https://books.toscrape.com/
Minimum required records: 1000


In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

In [8]:
os.makedirs("data", exist_ok=True)

print("Data directory is ready.")

Data directory is ready.


### Test Website Connection


In [9]:
url = CATALOGUE_URL + "page-1.html"

response = requests.get(url, timeout=REQUEST_TIMEOUT)

print("Status Code:", response.status_code)

Status Code: 200


### Inspect the Webpage Structure

We will parse the downloaded HTML page and inspect the structure of the first book.

This helps us identify the HTML elements that contain the information we want to extract.

In [10]:
soup = BeautifulSoup(response.text, "html.parser")

first_book = soup.select_one("article.product_pod")

print(first_book.prettify()[:3000])

<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



### Extract Data from a Single Book

In [11]:
book = soup.select_one("article.product_pod")

title = book.h3.a["title"]
price = book.select_one(".price_color").text.strip()
rating = book.select_one("p.star-rating")["class"][1]
availability = book.select_one(".availability").text.strip()

book_link = book.h3.a["href"]

print("Title:", title)
print("Price:", price)
print("Rating:", rating)
print("Availability:", availability)
print("Link:", book_link)

Title: A Light in the Attic
Price: Â£51.77
Rating: Three
Availability: In stock
Link: a-light-in-the-attic_1000/index.html


### Scrape All Books from One Page

The website contains multiple books on each catalog page.

We will now loop through all book containers on the first page and extract the required information from each book.

In [12]:
books = []

book_cards = soup.select("article.product_pod")

for book in book_cards:
    title = book.h3.a["title"]
    price = book.select_one(".price_color").text.strip()
    rating = book.select_one("p.star-rating")["class"][1]
    availability = book.select_one(".availability").text.strip()
    book_link = book.h3.a["href"]

    books.append({
        "Title": title,
        "Price": price,
        "Rating": rating,
        "Availability": availability,
        "Link": book_link
    })

print("Books extracted:", len(books))

Books extracted: 20


In [13]:
page1_df = pd.DataFrame(books)

page1_df.head()

,Title,Price,Rating,Availability,Link
0,A Light in the Attic,Â£51.77,Three,In stock,a-light-in-the-attic_1000/index.html
1,Tipping the Velvet,Â£53.74,One,In stock,tipping-the-velvet_999/index.html
2,Soumission,Â£50.10,One,In stock,soumission_998/index.html
3,Sharp Objects,Â£47.82,Four,In stock,sharp-objects_997/index.html
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,sapiens-a-brief-history-of-humankind_996/index...


### Scrape the Complete Book Catalog

The website contains 50 catalog pages with 20 books per page.

We will loop through all 50 pages and collect the available book information into a single dataset.

In [14]:
all_books = []

base_url = "https://books.toscrape.com/catalogue/page-{}.html"

for page in range(1, 51):

    url = base_url.format(page)

    response = requests.get(url, timeout=10)

    if response.status_code == 200:

        soup = BeautifulSoup(response.text, "html.parser")

        book_cards = soup.select("article.product_pod")

        for book in book_cards:

            title = book.h3.a["title"]
            price = book.select_one(".price_color").text.strip()
            rating = book.select_one("p.star-rating")["class"][1]
            availability = book.select_one(".availability").text.strip()
            book_link = book.h3.a["href"]

            all_books.append({
                "Title": title,
                "Price": price,
                "Rating": rating,
                "Availability": availability,
                "Link": book_link
            })

        print(f"Page {page}: {len(book_cards)} books extracted")

    else:
        print(f"Page {page}: Failed - Status Code {response.status_code}")

    time.sleep(0.5)

print("\nTotal books extracted:", len(all_books))

Page 1: 20 books extracted
Page 2: 20 books extracted
Page 3: 20 books extracted
Page 4: 20 books extracted
Page 5: 20 books extracted
Page 6: 20 books extracted
Page 7: 20 books extracted
Page 8: 20 books extracted
Page 9: 20 books extracted
Page 10: 20 books extracted
Page 11: 20 books extracted
Page 12: 20 books extracted
Page 13: 20 books extracted
Page 14: 20 books extracted
Page 15: 20 books extracted
Page 16: 20 books extracted
Page 17: 20 books extracted
Page 18: 20 books extracted
Page 19: 20 books extracted
Page 20: 20 books extracted
Page 21: 20 books extracted
Page 22: 20 books extracted
Page 23: 20 books extracted
Page 24: 20 books extracted
Page 25: 20 books extracted
Page 26: 20 books extracted
Page 27: 20 books extracted
Page 28: 20 books extracted
Page 29: 20 books extracted
Page 30: 20 books extracted
Page 31: 20 books extracted
Page 32: 20 books extracted
Page 33: 20 books extracted
Page 34: 20 books extracted
Page 35: 20 books extracted
Page 36: 20 books extracted
P

### Create the Raw Dataset

The scraped records will now be converted into a Pandas DataFrame.

In [15]:
raw_df = pd.DataFrame(all_books)

print("Dataset shape:", raw_df.shape)

raw_df.head()

Dataset shape: (1000, 5)


,Title,Price,Rating,Availability,Link
0,A Light in the Attic,Â£51.77,Three,In stock,a-light-in-the-attic_1000/index.html
1,Tipping the Velvet,Â£53.74,One,In stock,tipping-the-velvet_999/index.html
2,Soumission,Â£50.10,One,In stock,soumission_998/index.html
3,Sharp Objects,Â£47.82,Four,In stock,sharp-objects_997/index.html
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,sapiens-a-brief-history-of-humankind_996/index...


In [16]:
print("Total records:", len(raw_df))
print("Unique titles:", raw_df["Title"].nunique())

Total records: 1000
Unique titles: 999


In [17]:
duplicate_titles = raw_df[
    raw_df["Title"].duplicated(keep=False)
].sort_values("Title")

duplicate_titles

,Title,Price,Rating,Availability,Link
236,The Star-Touched Queen,Â£46.02,Five,In stock,the-star-touched-queen_764/index.html
358,The Star-Touched Queen,Â£32.30,Five,In stock,the-star-touched-queen_642/index.html


In [18]:
print("Duplicate records:", raw_df["Title"].duplicated().sum())

Duplicate records: 1


In [19]:
rawbook = raw_df.copy()

rawbook.to_csv("data/rawbook.csv", index=False, encoding="utf-8-sig")

print("Raw dataset saved successfully.")
print("Shape:", rawbook.shape)

Raw dataset saved successfully.
Shape: (1000, 5)


### Extract Detailed Information from Individual Book Pages

In [20]:
from urllib.parse import urljoin

In [ ]:
base_product_url = "https://books.toscrape.com/catalogue/"

detailed_books = []
failed_records = []

for index, row in raw_df.iterrows():

    product_url = urljoin(base_product_url, row["Link"])

    success = False

    for attempt in range(3):

        try:
            product_response = requests.get(
                product_url,
                timeout=20
            )

            if product_response.status_code == 200:

                product_soup = BeautifulSoup(
                    product_response.text,
                    "html.parser"
                )

                # Category
                breadcrumb = product_soup.select("ul.breadcrumb li")

                if len(breadcrumb) >= 3:
                    category = breadcrumb[2].get_text(strip=True)
                else:
                    category = None

                # Description
                description_tag = product_soup.select_one(
                    "#product_description + p"
                )

                if description_tag:
                    description = description_tag.get_text(strip=True)
                else:
                    description = None

                # Product information table
                product_info = {}

                for row_data in product_soup.select(
                    "table.table.table-striped tr"
                ):
                    header = row_data.find("th")
                    value = row_data.find("td")

                    if header and value:
                        key = header.get_text(strip=True)
                        value_text = value.get_text(strip=True)
                        product_info[key] = value_text

                detailed_books.append({
                    "Title": row["Title"],
                    "Price": row["Price"],
                    "Rating": row["Rating"],
                    "Availability": row["Availability"],
                    "Link": row["Link"],
                    "Category": category,
                    "Description": description,
                    "UPC": product_info.get("UPC"),
                    "Product Type": product_info.get("Product Type"),
                    "Tax": product_info.get("Tax"),
                    "Number Available": product_info.get("Availability"),
                    "Number of Reviews": product_info.get("Number of reviews")
                })

                success = True
                break

        except requests.exceptions.RequestException:
            print(f"Retry {attempt + 1}/3 for record {index}")
            time.sleep(2)

    if not success:
        failed_records.append(index)
        print(f"Failed after 3 attempts: record {index}")

    time.sleep(0.5)

print("\nDetailed records collected:", len(detailed_books))
print("Failed records:", len(failed_records))

### Create the Enriched Dataset

In [ ]:
detailed_df = pd.DataFrame(detailed_books)

print("Dataset shape:", detailed_df.shape)

detailed_df.head()

Dataset shape: (1000, 12)


,Title,Price,Rating,Availability,Link,Category,Description,UPC,Product Type,Tax,Number Available,Number of Reviews
0,A Light in the Attic,Â£51.77,Three,In stock,a-light-in-the-attic_1000/index.html,Poetry,It's hard to imagine a world without A Light i...,a897fe39b1053632,Books,Â£0.00,In stock (22 available),0
1,Tipping the Velvet,Â£53.74,One,In stock,tipping-the-velvet_999/index.html,Historical Fiction,"""Erotic and absorbing...Written with starling ...",90fa61229261140a,Books,Â£0.00,In stock (20 available),0
2,Soumission,Â£50.10,One,In stock,soumission_998/index.html,Fiction,"Dans une France assez proche de la nÃ´tre, un ...",6957f44c3847a760,Books,Â£0.00,In stock (20 available),0
3,Sharp Objects,Â£47.82,Four,In stock,sharp-objects_997/index.html,Mystery,"WICKED above her hipbone, GIRL across her hear...",e00eb4fd7b871a48,Books,Â£0.00,In stock (20 available),0
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,sapiens-a-brief-history-of-humankind_996/index...,History,From a renowned historian comes a groundbreaki...,4165285e1663650f,Books,Â£0.00,In stock (20 available),0


In [ ]:
print(detailed_df.columns.tolist())

['Title', 'Price', 'Rating', 'Availability', 'Link', 'Category', 'Description', 'UPC', 'Product Type', 'Tax', 'Number Available', 'Number of Reviews']


### Initial Data Quality Check

In [ ]:
missing_values = detailed_df.isnull().sum()

missing_values

Title                0
Price                0
Rating               0
Availability         0
Link                 0
Category             0
Description          2
UPC                  0
Product Type         0
Tax                  0
Number Available     0
Number of Reviews    0
dtype: int64

In [ ]:
detailed_df.iloc[0]

Title                                             A Light in the Attic
Price                                                          Â£51.77
Rating                                                           Three
Availability                                                  In stock
Link                              a-light-in-the-attic_1000/index.html
Category                                                        Poetry
Description          It's hard to imagine a world without A Light i...
UPC                                                   a897fe39b1053632
Product Type                                                     Books
Tax                                                             Â£0.00
Number Available                               In stock (22 available)
Number of Reviews                                                    0
Name: 0, dtype: str

In [ ]:
enriched_raw_file = "data/enriched_raw_books.csv"

detailed_df.to_csv(
    enriched_raw_file,
    index=False,
    encoding="utf-8-sig"
)

print("Enriched raw dataset saved successfully.")
print("File:", enriched_raw_file)
print("Shape:", detailed_df.shape)

Enriched raw dataset saved successfully.
File: data/enriched_raw_books.csv
Shape: (1000, 12)


In [ ]:
print("Total Records:", len(detailed_df))
print("Total Columns:", len(detailed_df.columns))
print("Failed Records:", len(failed_records))
print("Missing Descriptions:", detailed_df["Description"].isnull().sum())
print("Unique Product Links:", detailed_df["Link"].nunique())

Total Records: 1000
Total Columns: 12
Failed Records: 0
Missing Descriptions: 2
Unique Product Links: 1000


## Conclusion

The BookWise dataset was successfully collected from the publicly accessible Books to Scrape website using Python web scraping techniques.

A total of 1,000 book records were extracted across multiple pages, with information including title, price, rating, availability, category, description, UPC, product type, tax, stock availability, and other relevant attributes.

The collected raw dataset provides the foundation for the subsequent data cleaning, exploratory analysis, machine learning, and business intelligence stages of the BookWise project.

The extracted data was saved for further preprocessing and analysis.